# NB3 — Réduction, sélection et alternative probabiliste avec tuning intégré

Ce notebook couvre `P11` à `P15`. Les transformations de réduction ou de sélection sont ajustées une seule fois par pipeline dans la phase de tuning accéléré, puis seuls les algorithmes sont réglés.

Ce notebook conserve la **phase baseline sans optimisation**, puis ajoute une **phase d’optimisation accélérée**.
Le principe retenu est le suivant : pour chaque pipeline, on ajuste d’abord le **préprocesseur / vectoriseur une seule fois** sur un sous-ensemble d’apprentissage, puis on teste plusieurs réglages du **classifieur uniquement** sur les mêmes données déjà transformées. Cela réduit fortement le temps d’exécution.

Cette stratégie est très pratique pour explorer rapidement des réglages d’algorithmes, mais il faut bien comprendre qu’elle constitue une **optimisation accélérée**, plus pragmatique qu’une recherche entièrement relancée sur tout le pipeline à chaque itération.


In [ ]:
# Pour un run sur Colab

'''
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/Disaster-Tweets-NLP"
MODELS_DIR = f"{PROJECT_ROOT}/notebooks/models_training"

%cd "{MODELS_DIR}"

import sys
if MODELS_DIR not in sys.path:
    sys.path.append(MODELS_DIR)

print("Projet :", PROJECT_ROOT)
print("Dossier courant :", MODELS_DIR)
'''


'\nfrom google.colab import drive\ndrive.mount(\'/content/drive\')\n\nPROJECT_ROOT = "/content/drive/MyDrive/Disaster-Tweets-NLP"\nMODELS_DIR = f"{PROJECT_ROOT}/notebooks/models_training"\n\n%cd "{MODELS_DIR}"\n\nimport sys\nif MODELS_DIR not in sys.path:\n    sys.path.append(MODELS_DIR)\n\nprint("Projet :", PROJECT_ROOT)\nprint("Dossier courant :", MODELS_DIR)\n'

In [ ]:
# Installation éventuelle (décommente si nécessaire)
# !pip install pandas numpy scikit-learn scipy matplotlib gensim sentence-transformers openpyxl mlflow

import json
from collections import OrderedDict
from pathlib import Path

import pandas as pd
from mlflow_utils import (
    fit_evaluate_and_log_sklearn_pipeline,
    setup_mlflow_tracking,
)
from nlp_disaster_utils import (
    load_train_test_xy,
    round_results,
    save_results_bundle,
    seed_everything,
    stratified_validation_split,
)
from pipeline_tuning_utils import (
    compare_baseline_vs_tuned,
    evaluate_refit_outputs,
    fit_transform_preprocessor_once,
    log_tuning_run_to_mlflow,
    safe_scores,
    split_pipeline_preprocessor_estimator,
    tune_classifier_on_fixed_features,
)
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import Normalizer
from sklearn.svm import LinearSVC


seed_everything(42)


C:\Users\DELL\AppData\Local\Programs\Python\Python311\Lib\site-packages\pydantic\_internal\_fields.py:161: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [ ]:
DATA_DIR = "../../data/processed_data"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TEST_PATH = f"{DATA_DIR}/test.csv"

TEXT_COL = "text"
LABEL_COL = "target"
USE_AUX_TEXT_COLUMNS = False
LOWERCASE_TEXT = False
RANDOM_STATE = 42

OUTPUT_STEM = "NB3_reduction_selection_nb"
RESULTS_DIR = "../../outputs/NB3"
TUNING_OUTPUT_DIR = Path(RESULTS_DIR) / "tuning"
TUNING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Configuration MLflow
MLFLOW_EXPERIMENT_NAME = "DT_NB3_reduction_selection_nb"
MLFLOW_TRACKING_URI = Path("../../outputs/mlruns").resolve().as_uri()
MLFLOW_LOG_MODEL = False
USE_MLFLOW = True

tracking_uri = setup_mlflow_tracking(
    experiment_name=MLFLOW_EXPERIMENT_NAME,
    tracking_uri=MLFLOW_TRACKING_URI,
)
print("MLflow tracking URI :", tracking_uri)
print("MLflow experiment   :", MLFLOW_EXPERIMENT_NAME)

# Paramètres de tuning accéléré
VAL_SIZE_FOR_TUNING = 0.15
PRIMARY_TUNING_METRIC = "f1_pos"


MLflow tracking URI : file:///C:/Users/DELL/Documents/Classes/ISE2/ISE2_2026/SEM2/ML2/Projet/Disaster-Tweets-NLP/outputs/mlruns
MLflow experiment   : DT_NB3_reduction_selection_nb


C:\Users\DELL\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


Définition de tous les paramètres du notebook : chemins des données et des sorties, colonnes utilisées, et configuration MLflow (expérience `DT_NB3_reduction_selection_nb`, URI de tracking local). Le dossier `outputs/NB3/tuning` est créé automatiquement si absent. 15 % du train sont réservés pour la validation du tuning, avec le **F1 de la classe 1** (`f1_pos`) comme métrique de sélection des meilleurs hyperparamètres.

In [ ]:
df_train, X_train, y_train, df_test, X_test, y_test = load_train_test_xy(
    train_path=TRAIN_PATH,
    test_path=TEST_PATH,
    text_col=TEXT_COL,
    label_col=LABEL_COL,
    use_extra_cols=USE_AUX_TEXT_COLUMNS,
    lowercase=LOWERCASE_TEXT,
)

print("Taille train :", len(X_train))
print("Taille test  :", len(X_test))
print("\nDistribution des classes - train :")
print(y_train.value_counts(normalize=True).sort_index())
print("\nDistribution des classes - test :")
print(y_test.value_counts(normalize=True).sort_index())


Taille train : 9096
Taille test  : 2274

Distribution des classes - train :
target
0    0.814094
1    0.185906
Name: proportion, dtype: float64

Distribution des classes - test :
target
0    0.813984
1    0.186016
Name: proportion, dtype: float64


Chargement des jeux d'entraînement (9 096 tweets) et de test (2 274 tweets). La distribution des classes est quasi identique dans les deux splits : ~81 % Non-Disaster (classe 0) et ~19 % Disaster (classe 1).

In [ ]:
pipelines = OrderedDict({
    "P11_TFIDF_SVD_LogReg": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95)),
        ("svd", TruncatedSVD(n_components=300, random_state=42)),
        ("norm", Normalizer(copy=False)),
        ("clf", LogisticRegression(max_iter=2500, C=1.0)),
    ]),
    "P12_TFIDF_SVD_LinearSVC": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95)),
        ("svd", TruncatedSVD(n_components=300, random_state=42)),
        ("norm", Normalizer(copy=False)),
        ("clf", LinearSVC(C=1.0)),
    ]),
    "P13_TFIDF_SelectKBest_LogReg": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95)),
        ("select", SelectKBest(score_func=chi2, k=5000)),
        ("clf", LogisticRegression(max_iter=2500, C=1.0)),
    ]),
    "P14_TFIDF_SelectKBest_LinearSVC": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95)),
        ("select", SelectKBest(score_func=chi2, k=5000)),
        ("clf", LinearSVC(C=1.0)),
    ]),
    "P15_TFIDF_ComplementNB": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95)),
        ("clf", ComplementNB(alpha=0.5)),
    ]),
})


Définition des 5 pipelines (P11 à P15), tous basés sur un TF-IDF bigramme, mais avec des stratégies de réduction ou sélection différentes :

- **P11 & P12 (SVD)** : réduction dimensionnelle via TruncatedSVD (300 composantes) + normalisation, avec LogReg ou LinearSVC. SVD compresse l'espace TF-IDF en capturant les dimensions les plus informatives.
- **P13 & P14 (SelectKBest)** : sélection des 5 000 meilleures features par test chi2, qui garde uniquement les termes les plus discriminants pour la classification. Plus interprétable que SVD.
- **P15 (ComplementNB)** : pas de réduction — le Complement Naive Bayes est entraîné directement sur le TF-IDF. Variante de Naive Bayes conçue pour les classes déséquilibrées, elle modélise la classe complémentaire pour mieux discriminer la classe minoritaire.

## Phase 1 — Baselines sans optimisation

Cette première phase reproduit le benchmark initial : chaque pipeline est exécuté tel quel, avec ses paramètres de départ.

In [ ]:
resultats = []
baseline_failures = []

for nom_pipeline, pipeline in pipelines.items():
    print(f"Entraînement baseline -> {nom_pipeline}")
    display(pipeline)
    print("-" * 80)

    try:
        metrics = fit_evaluate_and_log_sklearn_pipeline(
            name=nom_pipeline,
            estimator=pipeline,
            X_train=X_train,
            X_test=X_test,
            y_train=y_train,
            y_test=y_test,
            notebook_name="REDUCTION",
            family_name="reduction_selection_nb",
            output_dir=RESULTS_DIR,
            log_model=MLFLOW_LOG_MODEL,
        )
        resultats.append(metrics)
    except Exception as exc:
        baseline_failures.append({"pipeline": nom_pipeline, "error": str(exc)})
        print(f"Échec baseline pour {nom_pipeline} : {exc}")

baseline_df = round_results(pd.DataFrame(resultats))
display(baseline_df)

if baseline_failures:
    print("\nPipelines baseline en échec :")
    display(pd.DataFrame(baseline_failures))


Entraînement baseline -> P11_TFIDF_SVD_LogReg


Pipeline(steps=[('vect',
                 TfidfVectorizer(max_df=0.95, min_df=2, ngram_range=(1, 2))),
                ('svd', TruncatedSVD(n_components=300, random_state=42)),
                ('norm', Normalizer(copy=False)),
                ('clf', LogisticRegression(max_iter=2500))])

--------------------------------------------------------------------------------
Entraînement baseline -> P12_TFIDF_SVD_LinearSVC


Pipeline(steps=[('vect',
                 TfidfVectorizer(max_df=0.95, min_df=2, ngram_range=(1, 2))),
                ('svd', TruncatedSVD(n_components=300, random_state=42)),
                ('norm', Normalizer(copy=False)), ('clf', LinearSVC())])

--------------------------------------------------------------------------------
Entraînement baseline -> P13_TFIDF_SelectKBest_LogReg


...

Pipeline(steps=[('vect',
                 TfidfVectorizer(max_df=0.95, min_df=2, ngram_range=(1, 2))),
                ('select',
                 SelectKBest(k=5000,
                             score_func=<function chi2 at 0x000001E6D8CBFD80>)),
                ('clf', LinearSVC())])

--------------------------------------------------------------------------------
Entraînement baseline -> P15_TFIDF_ComplementNB


Pipeline(steps=[('vect',
                 TfidfVectorizer(max_df=0.95, min_df=2, ngram_range=(1, 2))),
                ('clf', ComplementNB(alpha=0.5))])

--------------------------------------------------------------------------------


pipeline,P11_TFIDF_SVD_LogReg,P12_TFIDF_SVD_LinearSVC,P13_TFIDF_SelectKBest_LogReg,P14_TFIDF_SelectKBest_LinearSVC,P15_TFIDF_ComplementNB
train_accuracy,0.8830,0.8884,0.8883,0.9594,0.9454
train_precision_macro,0.8595,0.8570,0.9278,0.9666,0.9101
train_recall_macro,0.7224,0.7448,0.7037,0.8982,0.9092
train_f1_macro,0.7644,0.7833,0.7563,0.9278,0.9096
train_precision_weighted,0.8782,0.8828,0.8984,0.9601,0.9453
train_recall_weighted,0.8830,0.8884,0.8883,0.9594,0.9454
train_f1_weighted,0.8694,0.8781,0.8690,0.9578,0.9453
train_precision_class_0,0.8893,0.8981,0.8810,0.9563,0.9661
train_recall_class_0,0.9781,0.9734,0.9976,0.9957,0.9668
train_f1_class_0,0.9316,0.9342,0.9357,0.9756,0.9665


Entraînement et évaluation des 5 pipelines avec leurs paramètres par défaut, sans aucune optimisation. Les métriques sont loguées dans MLflow et les éventuels échecs sont capturés séparément.

Sur le **set de test**, en se concentrant sur la classe 1 (Disaster) :
- **P15 (ComplementNB)** ressort comme le meilleur compromis : meilleur F1 classe 1 (0.689), meilleure balanced accuracy (0.796) et bon ROC-AUC (0.901). Sa conception adaptée aux classes déséquilibrées lui donne un avantage naturel.
- **P14 (SelectKBest + LinearSVC)** obtient le meilleur recall classe 1 (0.549) et le meilleur PR-AUC (0.778), mais au prix d'un overfitting marqué (train_f1_class_1 = 0.880 vs test = 0.660).
- **P13 (SelectKBest + LogReg)** affiche la meilleure précision classe 1 (0.894) mais un recall très faible (0.319) — il est trop conservateur et rate beaucoup de vrais tweets disaster.
- **P11 et P12 (SVD)** sont les moins performants sur la classe 1, avec des F1 autour de 0.57-0.59, la réduction dimensionnelle semblant perdre de l'information discriminante.

⚠️ Le tuning devra prioritairement corriger le faible recall des pipelines SVD et SelectKBest+LogReg, notamment via `class_weight='balanced'`.

In [ ]:
save_results_bundle(pd.DataFrame(resultats), output_dir=RESULTS_DIR, stem=OUTPUT_STEM)
print(f"Fichiers CSV/XLSX baseline enregistrés dans {RESULTS_DIR}")


Fichiers CSV/XLSX baseline enregistrés dans ../../outputs/NB3


Les résultats baseline sont sauvegardés en CSV et XLSX dans `outputs/NB3`, prêts à être comparés avec les résultats du tuning en fin de notebook.

## Phase 2 — Tuning accéléré avec vectorisation unique par pipeline

Ici, pour chaque pipeline, on sépare le **préprocesseur** du **classifieur**. On ajuste le préprocesseur **une seule fois** sur un sous-ensemble d’apprentissage, puis on teste différentes combinaisons d’hyperparamètres du classifieur sur les mêmes données déjà vectorisées. Enfin, on réajuste le meilleur classifieur sur tout le train transformé une seule fois et on l’évalue sur train et test.

In [ ]:
X_fit, X_val, y_fit, y_val = stratified_validation_split(
    X_train,
    y_train,
    val_size=VAL_SIZE_FOR_TUNING,
    random_state=RANDOM_STATE,
)

print("Taille tuning-fit :", len(X_fit))
print("Taille tuning-val :", len(X_val))


Taille tuning-fit : 7731
Taille tuning-val : 1365


Création d'un sous-ensemble de validation stratifié (15 % du train) pour la phase de tuning : **7 731 tweets** pour ajuster le vectoriseur et les classifieurs, et **1 365 tweets** mis de côté pour comparer les combinaisons d'hyperparamètres. La stratification garantit que la distribution des classes est préservée dans les deux splits.

In [ ]:
classifier_param_grids = {
    "P11_TFIDF_SVD_LogReg": {
        "C": [0.25, 0.5, 1.0, 2.0],
        "class_weight": [None, "balanced"],
    },
    "P12_TFIDF_SVD_LinearSVC": {
        "C": [0.25, 0.5, 1.0, 2.0],
        "class_weight": [None, "balanced"],
    },
    "P13_TFIDF_SelectKBest_LogReg": {
        "C": [0.25, 0.5, 1.0, 2.0],
        "class_weight": [None, "balanced"],
    },
    "P14_TFIDF_SelectKBest_LinearSVC": {
        "C": [0.25, 0.5, 1.0, 2.0],
        "class_weight": [None, "balanced"],
    },
    "P15_TFIDF_ComplementNB": {
        "alpha": [0.1, 0.5, 1.0, 2.0],
    },
}


Grilles d'hyperparamètres à tester pour chaque classifieur. Pour LogReg et LinearSVC (P11 à P14), on explore 4 valeurs de régularisation `C` combinées avec ou sans `class_weight='balanced'`, soit 8 combinaisons par pipeline. Pour ComplementNB (P15), on teste uniquement 4 valeurs de lissage `alpha` — le rééquilibrage des classes n'étant pas un paramètre natif de ce classifieur.

In [ ]:
tuning_rows = []
tuning_failures = []
tuned_metrics_rows = []

for nom_pipeline, pipeline in pipelines.items():
    print("=" * 100)
    print(f"Tuning accéléré -> {nom_pipeline}")

    param_grid = classifier_param_grids.get(nom_pipeline)
    if param_grid is None:
        tuning_failures.append({"pipeline": nom_pipeline, "error": "Grille d'hyperparamètres absente"})
        print("Aucune grille trouvée.")
        continue

    try:
        preprocessor, clf_name, base_estimator = split_pipeline_preprocessor_estimator(pipeline)

        transformed = fit_transform_preprocessor_once(
            preprocessor=preprocessor,
            X_fit=X_fit,
            y_fit=y_fit,
            X_val=X_val,
        )

        best_params, tuning_results_df = tune_classifier_on_fixed_features(
            base_estimator=base_estimator,
            param_grid=param_grid,
            X_fit=transformed["X_fit_transformed"],
            y_fit=y_fit,
            X_val=transformed["X_val_transformed"],
            y_val=y_val,
            primary_metric=PRIMARY_TUNING_METRIC,
        )

        tuning_results_df.insert(0, "pipeline", nom_pipeline)
        tuning_results_df.insert(1, "classifier_name", clf_name)

        best_preprocessor_full, _, best_estimator_template = split_pipeline_preprocessor_estimator(pipeline)
        best_preprocessor_full.fit(X_train, y_train)
        X_train_vec = best_preprocessor_full.transform(X_train)
        X_test_vec = best_preprocessor_full.transform(X_test)

        best_estimator = base_estimator.set_params(**best_params)
        best_estimator.fit(X_train_vec, y_train)

        train_pred = best_estimator.predict(X_train_vec)
        test_pred = best_estimator.predict(X_test_vec)
        train_score = safe_scores(best_estimator, X_train_vec)
        test_score = safe_scores(best_estimator, X_test_vec)

        final_metrics = evaluate_refit_outputs(
            pipeline_name=nom_pipeline,
            y_train=y_train,
            y_test=y_test,
            train_pred=train_pred,
            test_pred=test_pred,
            train_score=train_score,
            test_score=test_score,
        )
        final_metrics["best_params"] = json.dumps(best_params, ensure_ascii=False)
        final_metrics["best_val_primary_score"] = float(tuning_results_df.iloc[0]["primary_score"])
        final_metrics["best_val_f1_class_1"] = float(tuning_results_df.iloc[0]["val_f1_class_1"])
        final_metrics["best_val_recall_class_1"] = float(tuning_results_df.iloc[0]["val_recall_class_1"])
        final_metrics["best_val_f1_macro"] = float(tuning_results_df.iloc[0]["val_f1_macro"])
        final_metrics["best_val_balanced_accuracy"] = float(tuning_results_df.iloc[0]["val_balanced_accuracy"])

        tuning_rows.append(tuning_results_df.iloc[0].to_dict() | {
            "pipeline": nom_pipeline,
            "best_params": json.dumps(best_params, ensure_ascii=False),
        })
        tuned_metrics_rows.append(final_metrics)

        tuning_results_path = TUNING_OUTPUT_DIR / f"{nom_pipeline}_tuning_validation_results.csv"
        tuning_results_df.to_csv(tuning_results_path, index=False)

        if USE_MLFLOW:
            log_tuning_run_to_mlflow(
                run_name=nom_pipeline,
                notebook_name="REDUCTION",
                family_name="reduction_selection_nb",
                best_params=best_params,
                tuning_results_df=tuning_results_df,
                final_metrics=final_metrics,
                output_dir=TUNING_OUTPUT_DIR,
            )

        print("Meilleurs paramètres :", best_params)
        print("Meilleur score de validation :", tuning_results_df.iloc[0]["primary_score"])

    except Exception as exc:
        tuning_failures.append({"pipeline": nom_pipeline, "error": str(exc)})
        print(f"Échec tuning pour {nom_pipeline} : {exc}")


Tuning accéléré -> P11_TFIDF_SVD_LogReg
Meilleurs paramètres : {'C': 2.0, 'class_weight': None}
Meilleur score de validation : 0.5995423340961098
Tuning accéléré -> P12_TFIDF_SVD_LinearSVC
Meilleurs paramètres : {'C': 2.0, 'class_weight': None}
Meilleur score de validation : 0.6157303370786517
Tuning accéléré -> P13_TFIDF_SelectKBest_LogReg
Meilleurs paramètres : {'C': 2.0, 'class_weight': 'balanced'}
Meilleur score de validation : 0.6654991243432574
Tuning accéléré -> P14_TFIDF_SelectKBest_LinearSVC
Meilleurs paramètres : {'C': 2.0, 'class_weight': 'balanced'}
Meilleur score de validation : 0.6784313725490196
Tuning accéléré -> P15_TFIDF_ComplementNB
Meilleurs paramètres : {'alpha': 0.1}
Meilleur score de validation : 0.6863468634686347


Pour chaque pipeline, le préprocesseur est ajusté une seule fois sur le split d'entraînement, puis le classifieur est optimisé sur les features déjà transformées. Le meilleur classifieur est ensuite ré-entraîné sur l'intégralité du train vectorisé, évalué sur le test, et loggué dans MLflow.

Les meilleurs paramètres trouvés :
- **P11 (SVD + LogReg)** : `C=2.0, class_weight=None` — score val : 0.600
- **P12 (SVD + LinearSVC)** : `C=2.0, class_weight=None` — score val : 0.616
- **P13 (SelectKBest + LogReg)** : `C=2.0, class_weight='balanced'` — score val : 0.665
- **P14 (SelectKBest + LinearSVC)** : `C=2.0, class_weight='balanced'` — score val : 0.678
- **P15 (ComplementNB)** : `alpha=0.1` — score val : **0.686**

Fait notable : les pipelines SVD (P11, P12) sont les seuls pour lesquels `class_weight=None` est retenu — la réduction dimensionnelle semble déjà lisser suffisamment l'espace des features pour que le rééquilibrage n'apporte pas de gain. **P15 obtient le meilleur score de validation**, confirmant l'avantage naturel du ComplementNB sur les classes déséquilibrées.

In [ ]:
tuning_best_df = round_results(pd.DataFrame(tuning_rows))
display(tuning_best_df)

tuned_results_df = round_results(pd.DataFrame(tuned_metrics_rows))
display(tuned_results_df)

if tuning_failures:
    print("\nPipelines tuning en échec :")
    display(pd.DataFrame(tuning_failures))


pipeline,P11_TFIDF_SVD_LogReg,P12_TFIDF_SVD_LinearSVC,P13_TFIDF_SelectKBest_LogReg,P14_TFIDF_SelectKBest_LinearSVC,P15_TFIDF_ComplementNB
classifier_name,clf,clf,clf,clf,clf
params,"{""C"": 2.0, ""class_weight"": null}","{""C"": 2.0, ""class_weight"": null}","{""C"": 2.0, ""class_weight"": ""balanced""}","{""C"": 2.0, ""class_weight"": ""balanced""}","{""alpha"": 0.1}"
primary_metric,f1_pos,f1_pos,f1_pos,f1_pos,f1_pos
primary_score,0.5995,0.6157,0.6655,0.6784,0.6863
val_accuracy,0.8718,0.8747,0.8601,0.8799,0.8755
val_precision_macro,0.8059,0.8088,0.7692,0.8014,0.7913
val_recall_macro,0.7345,0.7454,0.8169,0.8032,0.8202
val_f1_macro,0.7616,0.7704,0.7885,0.8023,0.8043
val_precision_weighted,0.8624,0.8663,0.8757,0.8802,0.8827
val_recall_weighted,0.8718,0.8747,0.8601,0.8799,0.8755


pipeline,P11_TFIDF_SVD_LogReg,P12_TFIDF_SVD_LinearSVC,P13_TFIDF_SelectKBest_LogReg,P14_TFIDF_SelectKBest_LinearSVC,P15_TFIDF_ComplementNB
train_accuracy,0.8880,0.8887,0.9224,0.9682,0.9612
train_precision_macro,0.8577,0.8569,0.8583,0.9357,0.9195
train_recall_macro,0.7425,0.7462,0.9103,0.9645,0.9629
train_f1_macro,0.7815,0.7844,0.8807,0.9492,0.9391
train_precision_weighted,0.8825,0.8831,0.9310,0.9700,0.9650
train_recall_weighted,0.8880,0.8887,0.9224,0.9682,0.9612
train_f1_weighted,0.8773,0.8786,0.9250,0.9687,0.9621
train_precision_class_0,0.8971,0.8986,0.9740,0.9904,0.9919
train_recall_class_0,0.9741,0.9731,0.9295,0.9704,0.9602
train_f1_class_0,0.9340,0.9344,0.9512,0.9803,0.9758


Affichage des résultats de validation (meilleurs paramètres par pipeline) et des métriques finales après ré-entraînement sur le train complet.

Sur le **set de test**, en se concentrant sur la classe 1 (Disaster) :
- **P14 (SelectKBest + LinearSVC)** obtient le meilleur F1 classe 1 (0.695) et la meilleure balanced accuracy (0.819) — le rééquilibrage `class_weight='balanced'` a fortement amélioré ce pipeline.
- **P15 (ComplementNB)** reste très compétitif avec un F1 classe 1 de 0.691, le meilleur recall classe 1 (0.731), le meilleur ROC-AUC (0.913) et PR-AUC (0.787).
- **P13 (SelectKBest + LogReg)** progresse significativement grâce au `class_weight='balanced'` (recall classe 1 : 0.766) mais au prix d'une précision plus faible (0.614).
- **P11 et P12 (SVD)** restent en retrait avec des F1 classe 1 autour de 0.58, confirmant que la réduction dimensionnelle par SVD perd de l'information utile pour détecter la classe minoritaire.

⚠️ L'overfitting reste marqué sur P13, P14 et P15 (train_f1_class_1 entre 0.81 et 0.92 vs test entre 0.68 et 0.70). **P14 et P15 sont les pipelines les plus solides de ce notebook**, avec le meilleur équilibre précision/recall sur le test.

In [ ]:
comparison_df = compare_baseline_vs_tuned(
    baseline_df=pd.DataFrame(resultats),
    tuned_df=pd.DataFrame(tuned_metrics_rows),
)
display(round_results(comparison_df))


,baseline_pipeline,baseline_train_accuracy,baseline_train_precision_macro,baseline_train_recall_macro,baseline_train_f1_macro,baseline_train_precision_weighted,baseline_train_recall_weighted,baseline_train_f1_weighted,baseline_train_precision_class_0,baseline_train_recall_class_0,baseline_train_f1_class_0,baseline_train_support_class_0,baseline_train_precision_class_1,baseline_train_recall_class_1,baseline_train_f1_class_1,baseline_train_support_class_1,baseline_train_balanced_accuracy,baseline_train_roc_auc,baseline_train_pr_auc,baseline_test_accuracy,baseline_test_precision_macro,baseline_test_recall_macro,baseline_test_f1_macro,baseline_test_precision_weighted,baseline_test_recall_weighted,baseline_test_f1_weighted,baseline_test_precision_class_0,baseline_test_recall_class_0,baseline_test_f1_class_0,baseline_test_support_class_0,baseline_test_precision_class_1,baseline_test_recall_class_1,baseline_test_f1_class_1,baseline_test_support_class_1,baseline_test_balanced_accuracy,baseline_test_roc_auc,baseline_test_pr_auc,tuned_pipeline,tuned_train_accuracy,tuned_train_precision_macro,tuned_train_recall_macro,tuned_train_f1_macro,tuned_train_precision_weighted,tuned_train_recall_weighted,tuned_train_f1_weighted,tuned_train_precision_class_0,tuned_train_recall_class_0,tuned_train_f1_class_0,tuned_train_support_class_0,tuned_train_precision_class_1,tuned_train_recall_class_1,tuned_train_f1_class_1,tuned_train_support_class_1,tuned_train_balanced_accuracy,tuned_train_roc_auc,tuned_train_pr_auc,tuned_test_accuracy,tuned_test_precision_macro,tuned_test_recall_macro,tuned_test_f1_macro,tuned_test_precision_weighted,tuned_test_recall_weighted,tuned_test_f1_weighted,tuned_test_precision_class_0,tuned_test_recall_class_0,tuned_test_f1_class_0,tuned_test_support_class_0,tuned_test_precision_class_1,tuned_test_recall_class_1,tuned_test_f1_class_1,tuned_test_support_class_1,tuned_test_balanced_accuracy,tuned_test_roc_auc,tuned_test_pr_auc,tuned_best_params,tuned_best_val_primary_score,tuned_best_val_f1_class_1,tuned_best_val_recall_class_1,tuned_best_val_f1_macro,tuned_best_val_balanced_accuracy,delta_test_f1_class_1,delta_test_recall_class_1,delta_test_f1_macro,delta_test_balanced_accuracy
0,P11_TFIDF_SVD_LogReg,0.8830,0.8595,0.7224,0.7644,0.8782,0.8830,0.8694,0.8893,0.9781,0.9316,7405.0000,0.8297,0.4666,0.5973,1691.0000,0.7224,0.8972,0.7405,0.8690,0.8085,0.7152,0.7472,0.8585,0.8690,0.8574,0.8881,0.9600,0.9226,1851.0000,0.7289,0.4704,0.5718,423.0000,0.7152,0.8852,0.6726,P11_TFIDF_SVD_LogReg,0.8880,0.8577,0.7425,0.7815,0.8825,0.8880,0.8773,0.8971,0.9741,0.9340,7405.0000,0.8182,0.5109,0.6290,1691.0000,0.7425,0.8992,0.7439,0.8672,0.7972,0.7242,0.7514,0.8568,0.8672,0.8579,0.8922,0.9519,0.9211,1851.0000,0.7023,0.4965,0.5817,423.0000,0.7242,0.8844,0.6721,"{""C"": 2.0, ""class_weight"": null}",0.5995,0.5995,0.5157,0.7616,0.7345,0.0099,0.0260,0.0042,0.0090
1,P12_TFIDF_SVD_LinearSVC,0.8884,0.8570,0.7448,0.7833,0.8828,0.8884,0.8781,0.8981,0.9734,0.9342,7405.0000,0.8159,0.5163,0.6324,1691.0000,0.7448,0.8998,0.7442,0.8676,0.7961,0.7290,0.7547,0.8577,0.8676,0.8592,0.8942,0.9498,0.9211,1851.0000,0.6981,0.5083,0.5882,423.0000,0.7290,0.8826,0.6684,P12_TFIDF_SVD_LinearSVC,0.8887,0.8569,0.7462,0.7844,0.8831,0.8887,0.8786,0.8986,0.9731,0.9344,7405.0000,0.8152,0.5192,0.6344,1691.0000,0.7462,0.9001,0.7449,0.8672,0.7950,0.7287,0.7541,0.8573,0.8672,0.8588,0.8941,0.9492,0.9209,1851.0000,0.6958,0.5083,0.5874,423.0000,0.7287,0.8824,0.6681,"{""C"": 2.0, ""class_weight"": null}",0.6157,0.6157,0.5394,0.7704,0.7454,-0.0008,0.0000,-0.0005,-0.0003
2,P13_TFIDF_SelectKBest_LogReg,0.8883,0.9278,0.7037,0.7563,0.8984,0.8883,0.8690,0.8810,0.9976,0.9357,7405.0000,0.9747,0.4098,0.5770,1691.0000,0.7037,0.9447,0.8668,0.8663,0.8792,0.6553,0.6969,0.8699,0.8663,0.8392,0.8643,0.9914,0.9235,1851.0000,0.8940,0.3191,0.4704,423.0000,0.6553,0.9076,0.7493,P13_TFIDF_SelectKBest_LogReg,0.9224,0.8583,0.9103,0.8807,0.9310,0.9224,0.9250,0.9740,0.9295,0.9512,7405.0000,0.7427,0.8912,0.8102,1691.000

Tableau comparatif baseline vs tuned. Les gains varient fortement selon les pipelines :

- **P13 (SelectKBest + LogReg)** : la plus grande transformation du notebook — gain de **+0.21 sur le F1 classe 1** et **+0.45 sur le recall classe 1** grâce à `class_weight='balanced'`. Un pipeline qui ratait presque tous les tweets disaster en baseline devient très compétitif après tuning.
- **P14 (SelectKBest + LinearSVC)** : gain solide de **+0.03 sur le F1 classe 1** et **+0.17 sur le recall**, avec une balanced accuracy en hausse de +0.06. Le rééquilibrage des classes a bien joué son rôle.
- **P15 (ComplementNB)** : progression modeste sur le F1 classe 1 (+0.003) mais un recall en nette hausse (+0.09) — ce pipeline était déjà le plus équilibré en baseline et le tuning n'a apporté qu'un ajustement fin.
- **P11 (SVD + LogReg)** : gain marginal sur toutes les métriques (+0.01 sur le F1 classe 1). La réduction SVD semble limiter le potentiel d'amélioration du tuning.
- **P12 (SVD + LinearSVC)** : quasi aucun changement (-0.001 sur le F1 classe 1) — ce pipeline a atteint ses limites avec la représentation SVD.

En conclusion, **P13 est la grande surprise de ce notebook**, avec un gain spectaculaire grâce au rééquilibrage des classes. **P14 et P15 restent les pipelines les plus solides**, avec le meilleur équilibre entre toutes les métriques sur le test.

In [ ]:
pd.DataFrame(tuning_rows).to_csv(TUNING_OUTPUT_DIR / f"{OUTPUT_STEM}_tuning_resume.csv", index=False)
pd.DataFrame(tuned_metrics_rows).to_csv(TUNING_OUTPUT_DIR / f"{OUTPUT_STEM}_tuned_final_results.csv", index=False)
pd.DataFrame(tuning_failures).to_csv(TUNING_OUTPUT_DIR / f"{OUTPUT_STEM}_tuning_failures.csv", index=False)

comparison_df.to_csv(TUNING_OUTPUT_DIR / f"{OUTPUT_STEM}_baseline_vs_tuned.csv", index=False)

print("Exports tuning enregistrés dans :", TUNING_OUTPUT_DIR)


Exports tuning enregistrés dans : ..\..\outputs\NB3\tuning


Tous les résultats du tuning sont sauvegardés dans `outputs/NB3/tuning` : le résumé des combinaisons testées, les métriques finales des modèles tunés, les éventuels échecs, et le tableau comparatif baseline vs tuned. Ces fichiers serviront de référence pour choisir le pipeline à emporter dans les prochains notebooks.

Le notebook contient désormais les deux temps de travail : un benchmark initial sans optimisation, puis une optimisation accélérée centrée sur les algorithmes.